# Iris Classification Project

**Nick Hollman**

Use this notebook to practice a basic machine learning workflow with the public Iris dataset. The model predicts one of three flower species from four measurements.

Before starting, complete 00_Create_Environment.ipynb and select this project's .venv_iris as the notebook kernel.

## 1. Read local settings

Run this cell to load the local .env settings and display the active Python interpreter.

The .env file contains RANDOM_STATE and TEST_SIZE. load_dotenv loads those settings into the process environment, and os.getenv reads them. Defaults apply if a setting is absent.

After editing .env during a notebook session, restart the kernel before running the notebook again. Check the printed interpreter path to confirm that it points to the project environment.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv

print("Notebook Python", sys.executable)
load_dotenv(Path.cwd() / ".env")
random_state = int(os.getenv("RANDOM_STATE", "42"))
test_size = float(os.getenv("TEST_SIZE", "0.20"))

print("Random seed", random_state)
print("Test proportion", test_size)


## 2. Load and split the data

Run this cell to load Iris from scikit-learn and split it into training and test sets. The data includes four measurements per flower and a species label.

train_test_split reserves part of the data for evaluation. stratify preserves the proportion of each species, and random_state keeps the split reproducible.


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)

x_train, x_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=test_size,
    random_state=random_state,
    stratify=iris.target,
)

print("Training rows", len(x_train))
print("Test rows", len(x_test))


Training rows 120
Test rows 30


## 3. Train a baseline model

Run this cell to train a Pipeline containing StandardScaler and LogisticRegression, then predict the species for the test rows.

The scaler learns from the training data. The pipeline applies that same transformation when predicting the test data.

fit trains the pipeline. predict returns the predicted species for each test row.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=random_state)),
])

model.fit(x_train, y_train)
predictions = model.predict(x_test)


## 4. Evaluate the results

Run this cell to display accuracy and macro F1 for the current model.

Accuracy measures the fraction of correct predictions. Macro F1 gives each species equal weight.

The initial verification run used 120 training rows and 30 test rows and reached about 93.33% accuracy with the default settings.


In [4]:
from sklearn.metrics import accuracy_score, f1_score

results = pd.DataFrame([{
    "training_rows": len(x_train),
    "test_rows": len(x_test),
    "accuracy": accuracy_score(y_test, predictions),
    "macro_f1": f1_score(y_test, predictions, average="macro"),
}])

results


,training_rows,test_rows,accuracy,macro_f1
0,120,30,0.933333,0.933333


## 5. Practice changing configuration

Change TEST_SIZE to 0.30 in the local .env, restart the kernel and rerun the notebook. Expect 105 training rows and 45 test rows. Restore 0.20 and repeat.

This exercise demonstrates how configuration affects the workflow. Repeatedly comparing test results is exploratory. Use a separate final holdout after using those results to choose settings.

Before committing, inspect the notebook and its outputs for private values. .gitignore excludes .env and .venv_iris. Remove any sensitive values displayed inside a saved notebook before sharing it.

To adapt this workflow to another project, replace the dataset and target, choose an appropriate split and metrics, and create a separate environment named for that project. Review the local changes before deciding when to commit and publish.
